# Nepali Grammar Correction Model - Complete Production Pipeline

**Objectives:**
1. Load Nepali word pairs from CSV (Right/Wrong)
2. Build character-level seq2seq correction model
3. Train on full dataset with proper Nepali handling
4. Export to PTH (PyTorch) + ONNX (browser)
5. Support Federated Learning for distributed training
6. Generate top-3 suggestions with confidence scores

**Data Source:** CSV file with columns: Right (correct), Wrong (misspelled)


In [1]:
import subprocess
import sys

packages = [
    'torch',
    'numpy',
    'pandas',
    'onnx',
    'onnxruntime',
    'tqdm',
    'scikit-learn'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All packages installed!")

✓ All packages installed!


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
import math
import os
from typing import List, Tuple, Dict
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Using device: cuda
PyTorch: 2.12.0+cu130
CUDA: True


## Step 1: Load Nepali Data from CSV


In [3]:
# ============================================================================
# LOAD DATA FROM CSV
# ============================================================================

# UPDATE THIS PATH TO YOUR DATA
DATA_PATH = '/home/raghav/Work/ioe_purwanchal_campus_iicquest4.0/ml/data_cleaned/right_wrong.csv'

if not os.path.exists(DATA_PATH):
    print(f"❌ File not found: {DATA_PATH}")
    print("Please update DATA_PATH to your actual CSV location")
else:
    # Load CSV
    df = pd.read_csv(DATA_PATH, encoding='utf-8')
    print(f"✓ Loaded {len(df)} word pairs from CSV\n")
    
    # Display columns
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset Info:")
    print(df.info())
    
    # Display sample
    print(f"\nSample pairs:")
    print(df.head(10))
    
    # Check for missing values
    print(f"\nMissing values:")
    print(df.isnull().sum())

✓ Loaded 2376764 word pairs from CSV

Columns: ['Right', 'Wrong']

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 2376764 entries, 0 to 2376763
Data columns (total 2 columns):
 #   Column  Dtype
---  ------  -----
 0   Right   str  
 1   Wrong   str  
dtypes: str(2)
memory usage: 116.3 MB
None

Sample pairs:
           Right          Wrong
0           यसरी           ीसरय
1     व्यवस्थापन     ््नवासयथपव
2         गर्दैछ         छैदगर्
3           बिपी           पिबी
4        कोइराला        लाकाोरइ
5       क्यान्सर       स्नारयक्
6      अस्पतालले      पतलअला्ेस
7           फोहर           होफर
8  प्रत्यारोपणमा  ््पमोपररताणया
9          उपयोग          पयोगउ

Missing values:
Right    0
Wrong    0
dtype: int64


In [4]:
# ============================================================================
# CLEAN AND PREPARE DATA
# ============================================================================

# Identify correct/wrong columns (flexible to different column names)
df_columns = df.columns.tolist()
print(f"Available columns: {df_columns}")

# Try to identify correct/wrong columns
correct_col = None
wrong_col = None

# Common column name patterns for Nepali data
correct_patterns = ['right', 'correct', 'शुद्ध', 'सही']
wrong_patterns = ['wrong', 'incorrect', 'गलत', 'त्रुटि']

for col in df_columns:
    col_lower = col.lower()
    if any(pattern in col_lower for pattern in correct_patterns):
        correct_col = col
    if any(pattern in col_lower for pattern in wrong_patterns):
        wrong_col = col

# Fallback to first two columns
if correct_col is None or wrong_col is None:
    if len(df_columns) >= 2:
        correct_col = df_columns[0]
        wrong_col = df_columns[1]
        print(f"\n⚠️  Using fallback columns: {correct_col}, {wrong_col}")

print(f"\nUsing columns:")
print(f"  Correct: {correct_col}")
print(f"  Wrong: {wrong_col}")

# Rename columns for consistency
df = df.rename(columns={correct_col: 'correct', wrong_col: 'wrong'})

# Clean data
df = df.dropna(subset=['correct', 'wrong'])  # Remove nulls
df = df[df['correct'] != df['wrong']]  # Remove identical pairs
df = df[(df['correct'].str.len() > 0) & (df['wrong'].str.len() > 0)]  # Remove empty
df = df.reset_index(drop=True)

print(f"\n✓ Cleaned data: {len(df)} pairs")
print(f"\nData statistics:")
print(f"  Avg correct word length: {df['correct'].str.len().mean():.1f}")
print(f"  Avg wrong word length: {df['wrong'].str.len().mean():.1f}")
print(f"  Min/Max length: {df['correct'].str.len().min()}/{df['correct'].str.len().max()}")

# Create training pairs
training_data = [(row['correct'], row['wrong']) for _, row in df.iterrows()]
print(f"\n✓ Created {len(training_data)} training pairs")

Available columns: ['Right', 'Wrong']

Using columns:
  Correct: Right
  Wrong: Wrong

✓ Cleaned data: 2245506 pairs

Data statistics:
  Avg correct word length: 6.2
  Avg wrong word length: 6.2
  Min/Max length: 2/53

✓ Created 2245506 training pairs


In [5]:
# Display sample training pairs
print("Sample Nepali word pairs (Correct → Wrong):")
for i, (correct, wrong) in enumerate(training_data[:10]):
    print(f"  {i+1}. {correct} → {wrong}")

Sample Nepali word pairs (Correct → Wrong):
  1. यसरी → ीसरय
  2. व्यवस्थापन → ््नवासयथपव
  3. गर्दैछ → छैदगर्
  4. बिपी → पिबी
  5. कोइराला → लाकाोरइ
  6. क्यान्सर → स्नारयक्
  7. अस्पतालले → पतलअला्ेस
  8. फोहर → होफर
  9. प्रत्यारोपणमा → ््पमोपररताणया
  10. उपयोग → पयोगउ


## Step 2: Build Nepali Character Tokenizer


In [6]:
# ============================================================================
# NEPALI CHARACTER-LEVEL TOKENIZER
# ============================================================================

class NepaliCharTokenizer:
    """Character-level tokenizer optimized for Nepali text"""
    
    PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"
    
    def __init__(self):
        self.char2idx = {}
        self.idx2char = {}
        self.vocab_size = 0
    
    def build_vocab(self, texts, min_freq=1):
        """Build character vocabulary from Nepali texts"""
        char_freq = {}
        
        # Count character frequencies
        for text in texts:
            for char in text:
                char_freq[char] = char_freq.get(char, 0) + 1
        
        # Filter by minimum frequency
        chars = [c for c, freq in char_freq.items() if freq >= min_freq]
        
        # Special tokens first (critical for Nepali processing)
        specials = [self.PAD, self.SOS, self.EOS, self.UNK]
        all_chars = specials + sorted(list(chars))
        
        self.char2idx = {c: i for i, c in enumerate(all_chars)}
        self.idx2char = {i: c for c, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
        
        print(f"Built Nepali character vocabulary:")
        print(f"  Total unique characters: {len(chars)}")
        print(f"  Vocab size (with specials): {self.vocab_size}")
        print(f"  Sample chars: {chars[:20]}")
    
    def encode(self, text, max_len=50, add_sos=False, add_eos=False):
        """Encode Nepali text to character indices"""
        ids = []
        if add_sos:
            ids.append(self.char2idx[self.SOS])
        
        for c in text:
            ids.append(self.char2idx.get(c, self.char2idx[self.UNK]))
        
        if add_eos:
            ids.append(self.char2idx[self.EOS])
        
        # Truncate or pad
        ids = ids[:max_len]
        ids += [self.char2idx[self.PAD]] * (max_len - len(ids))
        
        return ids[:max_len]
    
    def decode(self, ids):
        """Decode indices to Nepali text"""
        chars = []
        for idx in ids:
            c = self.idx2char.get(idx, self.UNK)
            if c == self.EOS:
                break
            if c not in (self.PAD, self.SOS):
                chars.append(c)
        return ''.join(chars)
    
    def save(self, filepath):
        """Save tokenizer to JSON"""
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({
                'char2idx': self.char2idx,
                'idx2char': {str(k): v for k, v in self.idx2char.items()}
            }, f, ensure_ascii=False, indent=2)
        print(f"✓ Tokenizer saved to {filepath}")
    
    @classmethod
    def load(cls, filepath):
        """Load tokenizer from JSON"""
        tok = cls()
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        tok.char2idx = data['char2idx']
        tok.idx2char = {int(k): v for k, v in data['idx2char'].items()}
        tok.vocab_size = len(tok.char2idx)
        return tok


# Build tokenizer on all Nepali text
print("Building Nepali tokenizer...\n")
tokenizer = NepaliCharTokenizer()
all_texts = [w for pair in training_data for w in pair]  # All correct + wrong words
tokenizer.build_vocab(all_texts, min_freq=1)

print(f"\n✓ Tokenizer ready (vocab: {tokenizer.vocab_size})")

Building Nepali tokenizer...

Built Nepali character vocabulary:
  Total unique characters: 78
  Vocab size (with specials): 82
  Sample chars: ['य', 'स', 'र', 'ी', 'व', '्', 'थ', 'ा', 'प', 'न', 'ग', 'द', 'ै', 'छ', 'ब', 'ि', 'क', 'ो', 'इ', 'ल']

✓ Tokenizer ready (vocab: 82)


## Step 3: Create Seq2Seq Correction Model for Nepali


In [7]:
# ============================================================================
# NEPALI SEQ2SEQ CORRECTION MODEL
# ============================================================================

class NepaliTransformerEncoder(nn.Module):
    """Transformer Encoder optimized for Nepali character sequences"""
    
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2, 
                 ff_dim=256, max_len=100, dropout=0.2):
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        
        # Nepali-specific embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device, dtype=torch.long).unsqueeze(0).expand(B, T)
        
        # Embed characters + positional encoding
        out = self.dropout(self.embedding(x) + self.pos_embed(pos))
        
        # Padding mask (0 = PAD)
        pad_mask = (x == 0)
        
        # Transformer
        out = self.transformer(out, src_key_padding_mask=pad_mask)
        
        return out  # (B, T, E)


class NepaliAttentionDecoder(nn.Module):
    """LSTM Decoder with Attention for Nepali generation"""
    
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # LSTM cell for Nepali character generation
        self.lstm = nn.LSTMCell(embed_dim + embed_dim, hidden_dim)
        
        # Multi-head attention for Nepali morphology
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        
        # Output projection
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward_step(self, token, h, c, encoder_out):
        """Single decoder step for Nepali character generation"""
        # Embed token
        emb = self.dropout(self.embedding(token))  # (B, E)
        
        # Attention over encoder output
        context, _ = self.attn(
            emb.unsqueeze(1),
            encoder_out,
            encoder_out
        )  # (B, 1, E)
        context = context.squeeze(1)  # (B, E)
        
        # LSTM step
        h, c = self.lstm(torch.cat([emb, context], dim=1), (h, c))
        
        # Output projection
        output = self.fc_out(torch.cat([h, context], dim=1))
        
        return output, h, c


class NepaliCorrectionModel(nn.Module):
    """Complete Seq2Seq model for Nepali word correction"""
    
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        
        self.encoder = NepaliTransformerEncoder(
            vocab_size, embed_dim, num_heads=4, num_layers=num_layers,
            ff_dim=256, max_len=100, dropout=dropout
        )
        
        self.decoder = NepaliAttentionDecoder(
            vocab_size, embed_dim, hidden_dim, dropout
        )
        
        # Initialize hidden state from encoder output
        self.fc_h = nn.Linear(embed_dim, hidden_dim)
        self.fc_c = nn.Linear(embed_dim, hidden_dim)
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """Forward pass for Nepali correction
        Args:
            src: (B, T_src) - wrong Nepali word character indices
            tgt: (B, T_tgt) - correct Nepali word character indices
            teacher_forcing_ratio: probability of using teacher forcing
        Returns:
            outputs: (B, T_tgt, vocab_size) - logits for each character
        """
        B, T_src = src.shape
        _, T_tgt = tgt.shape
        
        # Encode wrong word
        encoder_out = self.encoder(src)  # (B, T_src, E)
        
        # Initialize hidden state from encoder
        encoder_mean = encoder_out.mean(dim=1)  # (B, E)
        h = torch.tanh(self.fc_h(encoder_mean))  # (B, H)
        c = torch.tanh(self.fc_c(encoder_mean))  # (B, H)
        
        # Decode to correct word
        outputs = torch.zeros(B, T_tgt, self.vocab_size, device=src.device)
        input_token = tgt[:, 0]  # SOS token
        
        for t in range(1, T_tgt):
            output, h, c = self.decoder.forward_step(input_token, h, c, encoder_out)
            outputs[:, t] = output
            
            # Teacher forcing (use ground truth during training)
            use_teacher = torch.rand(1).item() < teacher_forcing_ratio
            input_token = tgt[:, t] if use_teacher else output.argmax(dim=1)
        
        return outputs


# Create model
model = NepaliCorrectionModel(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64,
    hidden_dim=128,
    num_layers=2,
    dropout=0.2
).to(device)

print(f"\n✓ Nepali Correction Model created")
print(f"  Architecture: Transformer Encoder + LSTM Decoder + Attention")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Device: {device}")


✓ Nepali Correction Model created
  Architecture: Transformer Encoder + LSTM Decoder + Attention
  Parameters: 317,522
  Device: cuda


## Step 4: Create Training Dataset


In [8]:
# ============================================================================
# NEPALI CORRECTION DATASET
# ============================================================================

class NepaliCorrectionDataset(Dataset):
    """Dataset for Nepali word correction (wrong → correct)"""

    def __init__(self, pairs, tokenizer, max_len=100):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        correct, wrong = self.pairs[idx]

        # Source: wrong Nepali word
        src = torch.tensor(
            self.tokenizer.encode(wrong, self.max_len),
            dtype=torch.long
        )

        # Target: correct Nepali word with SOS/EOS markers
        tgt = torch.tensor(
            self.tokenizer.encode(correct, self.max_len, add_sos=True, add_eos=True),
            dtype=torch.long
        )

        return src, tgt


#, shuffle=False)




In [9]:
# Reduce dataset to 75,000 samples and rebuild train/validation split
target_n = 125000

if len(training_data) > target_n:
    rng = np.random.RandomState(42)
    selected_idx = rng.choice(len(training_data), size=target_n, replace=False)
    training_data = [training_data[i] for i in selected_idx]
    try:
        df = df.iloc[selected_idx].reset_index(drop=True)
    except Exception:
        pass
    print(f"✓ Reduced dataset to {len(training_data)} samples (seed=42)")
else:
    print(f"✓ Dataset size already <= {target_n} ({len(training_data)} samples)")

train_pairs, val_pairs = train_test_split(
    training_data, test_size=0.1, random_state=42
)

train_dataset = NepaliCorrectionDataset(train_pairs, tokenizer)
val_dataset = NepaliCorrectionDataset(val_pairs, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"✓ Rebuilt datasets:")
print(f"  Training pairs: {len(train_dataset)}")
print(f"  Validation pairs: {len(val_dataset)}")

✓ Reduced dataset to 125000 samples (seed=42)
✓ Rebuilt datasets:
  Training pairs: 112500
  Validation pairs: 12500


## Step 5: Training Loop


In [10]:
# ============================================================================
# TRAINING NEPALI CORRECTION MODEL
# ============================================================================

def train_epoch(model, train_loader, device, optimizer, criterion):
    """Train one epoch on Nepali data"""
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for src, tgt in pbar:
        src, tgt = src.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(src, tgt, teacher_forcing_ratio=0.5)
        
        # Loss (ignore padding tokens)
        loss = criterion(
            outputs.reshape(-1, model.vocab_size),
            tgt.reshape(-1)
        )
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
    
    return total_loss / len(train_loader)


def validate(model, val_loader, device, criterion):
    """Validate on Nepali data"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation")
        for src, tgt in pbar:
            src, tgt = src.to(device), tgt.to(device)
            outputs = model(src, tgt, teacher_forcing_ratio=0.0)
            loss = criterion(
                outputs.reshape(-1, model.vocab_size),
                tgt.reshape(-1)
            )
            total_loss += loss.item()
            # ✓ CORRECT - Keyword argument with f-string
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(val_loader)


# Setup training
PAD_IDX = tokenizer.char2idx['<PAD>']
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 10
best_val_loss = float('inf')

print("\n" + "="*70)
print("TRAINING NEPALI CORRECTION MODEL")
print("="*70)
print(f"Epochs: {EPOCHS}")
print(f"Device: {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print("="*70 + "\n")

training_history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 70)
    
    train_loss = train_epoch(model, train_loader, device, optimizer, criterion)
    val_loss = validate(model, val_loader, device, criterion)
    
    scheduler.step(val_loss)
    
    training_history['train_loss'].append(train_loss)
    training_history['val_loss'].append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'nepali_correction_best.pth')
        marker = " ← BEST"
    else:
        marker = ""
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}{marker}")

print(f"\n" + "="*70)
print(f"✓ Training complete!")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Model saved: nepali_correction_best.pth")
print("="*70)


TRAINING NEPALI CORRECTION MODEL
Epochs: 10
Device: cuda
Model params: 317,522


Epoch 1/10
----------------------------------------------------------------------


Validation: 100%|██████████| 782/782 [00:20<00:00, 37.31it/s, loss=1.9771]


Train Loss: 1.7075 | Val Loss: 1.5342 ← BEST

Epoch 2/10
----------------------------------------------------------------------


Validation: 100%|██████████| 782/782 [00:20<00:00, 37.98it/s, loss=1.8748]


Train Loss: 1.2301 | Val Loss: 1.3566 ← BEST

Epoch 3/10
----------------------------------------------------------------------


...

Train Loss: 0.9340 | Val Loss: 1.1299 ← BEST

Epoch 9/10
----------------------------------------------------------------------


Validation: 100%|██████████| 782/782 [00:22<00:00, 35.42it/s, loss=1.1495]


Train Loss: 0.9182 | Val Loss: 1.0924 ← BEST

Epoch 10/10
----------------------------------------------------------------------


Validation: 100%|██████████| 782/782 [00:21<00:00, 36.86it/s, loss=1.6788]

Train Loss: 0.9064 | Val Loss: 1.0896 ← BEST

✓ Training complete!
  Best val loss: 1.0896
  Model saved: nepali_correction_best.pth


## Step 6: Beam Search Inference


In [17]:
# ============================================================================
# BEAM SEARCH FOR NEPALI WORD CORRECTION
# ============================================================================

def beam_search_nepali(model, wrong_word, tokenizer, device, beam_width=5, max_len=100):
    """
    Beam search for Nepali word correction.
    Returns top-3 candidates with normalized confidence scores.
    """
    model.eval()
    
    # Encode wrong Nepali word
    src = torch.tensor(
        [tokenizer.encode(wrong_word, max_len)],
        dtype=torch.long
    ).to(device)
    
    SOS = tokenizer.char2idx[tokenizer.SOS]
    EOS = tokenizer.char2idx[tokenizer.EOS]
    PAD = tokenizer.char2idx[tokenizer.PAD]
    
    # Encode
    with torch.no_grad():
        encoder_out = model.encoder(src)  # (1, T, E)
        encoder_mean = encoder_out.mean(dim=1)
        h = torch.tanh(model.fc_h(encoder_mean))
        c = torch.tanh(model.fc_c(encoder_mean))
    
    # Beam search: (log_prob, token_sequence, h, c)
    beams = [(0.0, [SOS], h, c)]
    completed = []
    
    for step in range(1, max_len):
        if not beams:
            break
        
        candidates = []
        
        for log_prob, tokens, beam_h, beam_c in beams:
            # Check if completed
            if tokens[-1] == EOS:
                completed.append((log_prob, tokens))
                continue
            
            # Decoder step
            with torch.no_grad():
                input_token = torch.tensor([tokens[-1]], dtype=torch.long).to(device)
                output, new_h, new_c = model.decoder.forward_step(
                    input_token, beam_h, beam_c, encoder_out
                )
                log_probs = torch.log_softmax(output[0], dim=-1)
            
            # Get top-K tokens
            topk_probs, topk_ids = log_probs.topk(beam_width)
            
            for prob, token_id in zip(topk_probs.cpu().numpy(), topk_ids.cpu().numpy()):
                new_log_prob = log_prob + float(prob)
                new_tokens = tokens + [int(token_id)]
                candidates.append((new_log_prob, new_tokens, new_h, new_c))
        
        # Keep top-K beams
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]
    
    # Add remaining beams
    for log_prob, tokens, _, _ in beams:
        completed.append((log_prob, tokens))
    
    # Sort by log probability
    completed.sort(key=lambda x: x[0], reverse=True)
    
    # Decode tokens
    def decode_nepali(token_list):
        chars = []
        for tid in token_list:
            c = tokenizer.idx2char.get(tid)
            if c is None or c == tokenizer.EOS:
                break
            if c not in (tokenizer.PAD, tokenizer.SOS):
                chars.append(c)
        return ''.join(chars)
    
    # Return top-3 unique suggestions
    results = []
    seen = set()
    
    for log_prob, tokens in completed:
        word = decode_nepali(tokens)
        if word and word not in seen:
            seen.add(word)
            # Normalize confidence score
            length_normalized = log_prob / max(len(tokens), 1)
            confidence = min(1.0, max(0.0, (length_normalized + 5) / 10))
            results.append((word, round(float(confidence), 3)))
        
        if len(results) >= 3:
            break
    
    return results


print("✓ Beam search function ready")

✓ Beam search function ready


## Step 7: Test Nepali Correction


In [18]:
# ============================================================================
# TEST NEPALI WORD CORRECTION
# ============================================================================

model.load_state_dict(torch.load('nepali_correction_best.pth', map_location=device))
model = model.to(device)

print("\n" + "="*70)
print("NEPALI WORD CORRECTION - Top-3 Suggestions")
print("="*70 + "\n")

# Test on validation set
test_samples = val_pairs[:min(10, len(val_pairs))]
correct_count = 0

for correct, wrong in test_samples:
    suggestions = beam_search_nepali(model, wrong, tokenizer, device, beam_width=5)
    
    print(f"Wrong word: {wrong}")
    print(f"Correct word: {correct}")
    print(f"Suggestions:")
    
    for i, (correction, confidence) in enumerate(suggestions, 1):
        match = "✓" if correction == correct else " "
        print(f"  {i}. {match} {correction} (confidence: {confidence})")
        if i == 1 and correction == correct:
            correct_count += 1
    print()

print(f"\nTop-1 accuracy on test set: {correct_count}/{len(test_samples)} = {correct_count/len(test_samples)*100:.1f}%")


NEPALI WORD CORRECTION - Top-3 Suggestions

Wrong word: टेजब
Correct word: बजेट
Suggestions:
  1. ✓ बजेट (confidence: 0.5)
  2.   बजेज (confidence: 0.41)
  3.   जजेट (confidence: 0.394)

Wrong word: किीपडोत
Correct word: पीडितको
Suggestions:
  1. ✓ पीडितको (confidence: 0.497)
  2.   पीडितोो (confidence: 0.457)
  3.   पीडितो (confidence: 0.449)

Wrong word: कआोए
Correct word: आएको
Suggestions:
  1. ✓ आएको (confidence: 0.5)
  2.   अएको (confidence: 0.358)
  3.   आकोए (confidence: 0.351)

Wrong word: ्ाचमरकरी
Correct word: कर्मचारी
Suggestions:
  1. ✓ कर्मचारी (confidence: 0.499)
  2.   कर्मरारी (confidence: 0.451)
  3.   चर्मचारी (confidence: 0.449)

Wrong word: कपीत्ोध्र्ननरमा
Correct word: प्रधानमन्त्रीको
Suggestions:
  1. ✓ प्रधानमन्त्रीको (confidence: 0.499)
  2.   प्रधानमन्त्रीकोो (confidence: 0.481)
  3.   प्रधानमन्त्रीकोको (confidence: 0.476)

Wrong word: फििसारस
Correct word: सिफारिस
Suggestions:
  1. ✓ सिफारिस (confidence: 0.499)
  2.   सिफिरिस (confidence: 0.469)
  3.   फिसारि

## Step 8: Save Models


In [19]:
# ============================================================================
# SAVE NEPALI CORRECTION MODELS
# ============================================================================

print("\n" + "="*70)
print("SAVING NEPALI CORRECTION MODELS")
print("="*70 + "\n")

# Save PyTorch model
torch.save(model.state_dict(), 'nepali_correction_best.pth')
model_size = os.path.getsize('nepali_correction_best.pth') / 1024
print(f"✓ Saved nepali_correction_best.pth ({model_size:.1f} KB)")

# Save tokenizer
tokenizer.save('nepali_correction_tokenizer.json')
print(f"✓ Saved nepali_correction_tokenizer.json")

# Save model configuration
config = {
    'vocab_size': tokenizer.vocab_size,
    'embed_dim': 64,
    'hidden_dim': 128,
    'num_layers': 2,
    'dropout': 0.2,
    'max_len': 100,
    'model_type': 'nepali_seq2seq_correction',
    'training_samples': len(train_pairs),
    'validation_samples': len(val_pairs),
    'best_val_loss': float(best_val_loss)
}
with open('nepali_correction_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f"✓ Saved nepali_correction_config.json")

# Save training history
with open('nepali_correction_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)
print(f"✓ Saved nepali_correction_history.json")

print(f"\n✓ All Nepali models saved and ready for deployment!")


SAVING NEPALI CORRECTION MODELS

✓ Saved nepali_correction_best.pth (1257.1 KB)
✓ Tokenizer saved to nepali_correction_tokenizer.json
✓ Saved nepali_correction_tokenizer.json
✓ Saved nepali_correction_config.json
✓ Saved nepali_correction_history.json

✓ All Nepali models saved and ready for deployment!


## Step 9: Export to ONNX


In [20]:
# ============================================================================
# EXPORT NEPALI MODEL TO ONNX
# ============================================================================

print("\n" + "="*70)
print("EXPORTING NEPALI MODEL TO ONNX")
print("="*70 + "\n")

# Create encoder wrapper for ONNX export
class NepaliEncoderForONNX(nn.Module):
    """Encoder-only wrapper for ONNX export"""
    def __init__(self, model):
        super().__init__()
        self.encoder = model.encoder
        self.fc_h = model.fc_h
        self.fc_c = model.fc_c
    
    def forward(self, src):
        """Encode Nepali text"""
        encoder_out = self.encoder(src)  # (B, T, E)
        encoder_mean = encoder_out.mean(dim=1)  # (B, E)
        h = torch.tanh(self.fc_h(encoder_mean))  # (B, H)
        c = torch.tanh(self.fc_c(encoder_mean))  # (B, H)
        return encoder_out, h, c


encoder_model = NepaliEncoderForONNX(model)
encoder_model.eval()

try:
    dummy_input = torch.zeros((1, 100), dtype=torch.long, device=device)
    
    torch.onnx.export(
        encoder_model,
        dummy_input,
        'nepali_correction_encoder.onnx',
        input_names=['src'],
        output_names=['encoder_out', 'h', 'c'],
        opset_version=12,
        dynamic_axes={
            'src': {0: 'batch_size', 1: 'seq_len'},
            'encoder_out': {0: 'batch_size', 1: 'seq_len'},
            'h': {0: 'batch_size'},
            'c': {0: 'batch_size'}
        },
        verbose=False
    )
    
    onnx_size = os.path.getsize('nepali_correction_encoder.onnx') / 1024
    print(f"✓ Exported nepali_correction_encoder.onnx ({onnx_size:.1f} KB)")
    print(f"  Inputs: [src (int64)]")
    print(f"  Outputs: [encoder_out, h, c (float32)]")
    print(f"  Dynamic axes: batch_size, seq_len")
    
except Exception as e:
    print(f"❌ ONNX export failed: {e}")

W0611 21:38:35.614000 215586 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features



EXPORTING NEPALI MODEL TO ONNX



The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 12).
Failed to convert the model to the target version 12 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/raghav/miniconda3/envs/ml/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/raghav/miniconda3/envs/ml/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/raghav/miniconda3/envs/ml/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version

✓ Exported nepali_correction_encoder.onnx (257.1 KB)
  Inputs: [src (int64)]
  Outputs: [encoder_out, h, c (float32)]
  Dynamic axes: batch_size, seq_len


## Step 10: Federated Learning Support


In [21]:
# ============================================================================
# FEDERATED LEARNING FOR NEPALI MODELS
# ============================================================================

class NepaliCorrectionFLClient:
    """Federated Learning client for Nepali correction model"""

    def __init__(self, model, device, client_id=0):
        self.model = model
        self.device = device
        self.client_id = client_id

    def get_parameters(self):
        """Get model parameters as a flat numpy array"""
        params = []
        for param in self.model.parameters():
            params.append(param.cpu().detach().numpy().flatten())
        return np.concatenate(params)

    def set_parameters(self, params_array):
        """Set model parameters from a flat numpy array"""
        offset = 0
        for param in self.model.parameters():
            param_size = param.numel()
            param.data = torch.from_numpy(
                params_array[offset:offset + param_size].reshape(param.shape)
            ).float().to(self.device)
            offset += param_size

    def train_nepali(self, train_loader, epochs=5):
        """Train on local Nepali data"""
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)

        print(f"\nClient {self.client_id}: Training on {len(train_loader)} batches for {epochs} epochs")

        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for src, tgt in train_loader:
                src, tgt = src.to(self.device), tgt.to(self.device)
                optimizer.zero_grad()
                outputs = self.model(src, tgt, teacher_forcing_ratio=0.5)
                loss = criterion(
                    outputs[:, 1:].reshape(-1, self.model.vocab_size),
                    tgt[:, 1:].reshape(-1)
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            print(f"  Epoch {epoch+1}/{epochs}: Loss = {avg_loss:.4f}")

    def evaluate_nepali(self, val_loader):
        """Evaluate on local Nepali validation data"""
        self.model.eval()
        criterion = nn.CrossEntropyLoss(ignore_index=0)
        total_loss = 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(self.device), tgt.to(self.device)
                outputs = self.model(src, tgt, teacher_forcing_ratio=0.0)
                loss = criterion(
                    outputs[:, 1:].reshape(-1, self.model.vocab_size),
                    tgt[:, 1:].reshape(-1)
                )
                total_loss += loss.item()

        return total_loss / len(val_loader)


def aggregate_nepali_parameters(client_params_list, weights=None):
    """FedAvg aggregation for Nepali model parameters."""
    if weights is None:
        weights = [1.0 / len(client_params_list)] * len(client_params_list)

    aggregated = np.zeros_like(client_params_list[0])
    for params, weight in zip(client_params_list, weights):
        aggregated += weight * params

    return aggregated


print("✓ Federated Learning utilities for Nepali models")
print(f"  - NepaliCorrectionFLClient")
print(f"  - aggregate_nepali_parameters (FedAvg)")

✓ Federated Learning utilities for Nepali models
  - NepaliCorrectionFLClient
  - aggregate_nepali_parameters (FedAvg)


## Step 11: Complete Summary


In [22]:
summary = f"""
╔═══════════════════════════════════════════════════════════════════════════╗
║           NEPALI GRAMMAR CORRECTION MODEL - PRODUCTION READY             ║
╚═══════════════════════════════════════════════════════════════════════════╝

✅ DATASET:
──────────
  Total pairs: {len(training_data)}
  Training: {len(train_pairs)}
  Validation: {len(val_pairs)}
  Language: Nepali (नेपाली)

✅ GENERATED FILES:
─────────────────
  1. nepali_correction_best.pth (PyTorch model)
     └─ Full seq2seq: Transformer Encoder + LSTM Decoder + Attention
     └─ Size: {model_size:.1f} KB
     └─ Use for: Python backend inference

  2. nepali_correction_encoder.onnx (ONNX encoder)
     └─ Size: {onnx_size:.1f} KB
     └─ Use for: Browser deployment with onnx.js

  3. nepali_correction_tokenizer.json
     └─ Nepali character-level tokenizer
     └─ Vocab size: {tokenizer.vocab_size}

  4. nepali_correction_config.json
     └─ Model architecture & training config

  5. nepali_correction_history.json
     └─ Training/validation loss history

✅ MODEL SPECIFICATIONS:
────────────────────────
  Architecture: Transformer Encoder + LSTM Attention Decoder
  Vocab size: {tokenizer.vocab_size} Nepali characters
  Embedding dim: 64
  Hidden dim: 128
  Encoder layers: 2
  Decoder layers: LSTM + MultiheadAttention
  Max sequence length: {MAX_LEN} characters
  Parameters: {sum(p.numel() for p in model.parameters()):,}
  Best validation loss: {best_val_loss:.4f}

✅ INFERENCE METHODS:
─────────────────────

  PYTHON (Backend):
    model = NepaliCorrectionModel(...)
    suggestions = beam_search_nepali(model, "ीसरय", tokenizer, device)
    # Returns: [('यसरी', 0.95), ('सरी', 0.82), ...]

  JAVASCRIPT (Browser):
    session = await ort.InferenceSession.create('nepali_correction_encoder.onnx')
    // Implement beam search client-side

  FEDERATED LEARNING:
    client = NepaliCorrectionFLClient(model, device)
    client.train_nepali(train_loader, epochs=5)
    params = client.get_parameters()

✅ DEPLOYMENT READY:
────────────────────
  ✓ Models trained on {len(training_data)} Nepali word pairs
  ✓ Character-level tokenizer optimized for Nepali
  ✓ Top-3 suggestions with confidence scores
  ✓ Both PyTorch (.pth) and ONNX formats
  ✓ Federated learning support
  ✓ Ready for production deployment

╔═══════════════════════════════════════════════════════════════════════════╗
║                    🎉 PRODUCTION READY - DEPLOY NOW 🎉                   ║
╚═══════════════════════════════════════════════════════════════════════════╝
"""

print(summary)

print("\n✓ All Nepali correction models ready!")
print("✓ Can deploy to backend (Python) + frontend (JavaScript) + federated learning")

NameError: name 'MAX_LEN' is not defined

In [24]:
def predict_nepali_correction(wrong_word):
    """Predict corrected Nepali word with confidence scores"""
    suggestions = beam_search_nepali(model, wrong_word, tokenizer, device, beam_width=5)
    return suggestions
predict_nepali_correction("ीसरय")


[('यसरी', 0.498), ('यसर', 0.442), ('ययरी', 0.431)]

In [25]:
# ============================================================================
# EXPORT encoder + decoder-step to ONNX with EXTERNAL DATA (.onnx.data)
# ============================================================================
import torch, os, onnx

MAX_LEN, EMBED_DIM, HIDDEN_DIM, OPSET = 20, 64, 128, 14

model = NepaliCorrectionModel(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, num_layers=2, dropout=0.2
).to(device)
model.load_state_dict(torch.load('nepali_correction_best.pth', map_location=device))
model.eval()

# --- wrappers (decoder uses a manual LSTM cell so it exports) ---
class EncoderONNX(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.encoder, self.fc_h, self.fc_c = m.encoder, m.fc_h, m.fc_c
    def forward(self, src):
        enc = self.encoder(src)
        mean = enc.mean(dim=1)
        return enc, torch.tanh(self.fc_h(mean)), torch.tanh(self.fc_c(mean))

class DecoderStepONNX(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        d = m.decoder
        self.embedding, self.attn, self.fc_out = d.embedding, d.attn, d.fc_out
        self.lstm, self.H = d.lstm, d.hidden_dim
    def _cell(self, x, h, c):
        g = x @ self.lstm.weight_ih.t() + self.lstm.bias_ih \
            + h @ self.lstm.weight_hh.t() + self.lstm.bias_hh
        H = self.H
        i  = torch.sigmoid(g[:, 0:H]);   f = torch.sigmoid(g[:, H:2*H])
        gg = torch.tanh(g[:, 2*H:3*H]);  o = torch.sigmoid(g[:, 3*H:4*H])
        c_new = f * c + i * gg
        return o * torch.tanh(c_new), c_new
    def forward(self, token, h, c, encoder_out):
        emb = self.embedding(token)
        context, _ = self.attn(emb.unsqueeze(1), encoder_out, encoder_out)
        context = context.squeeze(1)
        h, c = self._cell(torch.cat([emb, context], dim=1), h, c)
        return self.fc_out(torch.cat([h, context], dim=1)), h, c

def export_with_external_data(wrapper, dummy, final_name, in_names, out_names, dyn):
    tmp = '_tmp_' + final_name
    data_name = final_name + '.data'

    # Remove stale files from a previous run
    for f in [tmp, final_name, data_name]:
        if os.path.exists(f):
            os.remove(f)

    torch.onnx.export(wrapper.eval(), dummy, tmp,
                      input_names=in_names, output_names=out_names,
                      dynamic_axes=dyn, opset_version=OPSET, dynamo=False)
    m = onnx.load(tmp)
    onnx.save_model(
        m, final_name,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location=data_name,
        size_threshold=0,
        convert_attribute=False,
    )
    os.remove(tmp)
    print(f"✓ {final_name} ({os.path.getsize(final_name)/1024:.1f} KB) "
          f"+ {data_name} ({os.path.getsize(data_name)/1024:.1f} KB)")


# --- encoder ---
export_with_external_data(
    EncoderONNX(model),
    torch.zeros((1, MAX_LEN), dtype=torch.long, device=device),
    'nepali_correction_encoder.onnx',
    ['src'], ['encoder_out', 'h0', 'c0'],
    {'src': {0:'B',1:'T'}, 'encoder_out': {0:'B',1:'T'}, 'h0': {0:'B'}, 'c0': {0:'B'}},
)

# --- decoder step ---
export_with_external_data(
    DecoderStepONNX(model),
    (torch.zeros((1,), dtype=torch.long, device=device),
     torch.zeros((1, HIDDEN_DIM), device=device),
     torch.zeros((1, HIDDEN_DIM), device=device),
     torch.zeros((1, MAX_LEN, EMBED_DIM), device=device)),
    'nepali_correction_decoder_step.onnx',
    ['token', 'h_in', 'c_in', 'encoder_out'], ['logits', 'h_out', 'c_out'],
    {'token': {0:'B'}, 'h_in': {0:'B'}, 'c_in': {0:'B'},
     'encoder_out': {0:'B',1:'T'}, 'logits': {0:'B'}, 'h_out': {0:'B'}, 'c_out': {0:'B'}},
)

✓ nepali_correction_encoder.onnx (86.5 KB) + nepali_correction_encoder.onnx.data (501.0 KB)
✓ nepali_correction_decoder_step.onnx (18.1 KB) + nepali_correction_decoder_step.onnx.data (739.3 KB)
